### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
%%capture
# Install latest transformers for Gemma 3N
!pip install --no-deps --upgrade timm # Only for Gemma 3N

### Unsloth

We'll use `FastModel` to load model. The model that we're gonna use is `gemma-3n-E2B-it-unsloth-bnb-4bit`. This is the quantized 4 bit version of the original `gemma-3n-E2B-it`. We are using this model because the original model need too many RAM.

In [ ]:
from unsloth import FastModel
import torch
from huggingface_hub import snapshot_download

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, processor = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit",
    dtype = None, # None for auto detection
    max_seq_length = 1024, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.8: Fast Gemma3N patching. Transformers: 4.53.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3N does not support SDPA - switching to eager!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/469M [00:00<?, ?B/s]

...

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

# Gemma 3N can process Text, Vision and Audio!

We will use Gemma 3N's recommended settings of `temperature = 1.0, top_p = 0.95, top_k = 64` for inference. For this example we use `do_sample=False` for ASR.

In [ ]:
from transformers import TextStreamer
# Helper function for inference
def do_gemma_3n_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **processor.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        do_sample=False,
        streamer = TextStreamer(processor, skip_prompt = True),
    )

<h3>Let's Evaluate Gemma 3N Baseline Performance on Indonesia Transcription</h2>

In [ ]:
from huggingface_hub import notebook_login; notebook_login()

In [ ]:
from datasets import load_dataset,Audio,concatenate_datasets

dataset = load_dataset("mozilla-foundation/common_voice_17_0", "id", split="train")

# Select a single audio sample to reserve for testing.
# This index is chosen from the full dataset before we create the smaller training split.
test_audio = dataset[4570]

dataset = dataset.select(range(300))

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

common_voice_17_0.py:   0%|          | 0.00/8.19k [00:00<?, ?B/s]

languages.py:   0%|          | 0.00/3.92k [00:00<?, ?B/s]

release_stats.py:   0%|          | 0.00/132k [00:00<?, ?B/s]

The repository for mozilla-foundation/common_voice_17_0 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/mozilla-foundation/common_voice_17_0.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


n_shards.json:   0%|          | 0.00/17.5k [00:00<?, ?B/s]

...


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 13402it [00:00, 134010.86it/s]
Reading metadata...: 29508it [00:00, 135600.76it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 2605it [00:00, 78704.00it/s]


Generating validated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 7854it [00:00, 78534.08it/s]
Reading metadata...: 16423it [00:00, 82741.04it/s]
Reading metadata...: 26108it [00:00, 81770.41it/s]


In [ ]:
from IPython.display import Audio, display
print(test_audio['sentence'])
Audio(test_audio['audio']['array'],rate=test_audio['audio']['sampling_rate'])

Anda telah menyebabkan Wali Kota dan Pullman berada pada posisi yang kurang menguntungkan.


<h3>Let's Try to do inference on test audio and see what the model outputs!</h2>
For some reason the model output something nonsensical. We're gonna fix that using LoRA.

In [ ]:
messages = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": "You are an assistant that transcribes speech accurately.",
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": test_audio['audio']['array']},
                    {"type": "text", "text": "Please transcribe this audio."}
                ]
            }
        ]

do_gemma_3n_inference(messages, max_new_tokens = 256)

R
<end_of_turn>


# Let's finetune Gemma 3N!

You can finetune the vision and text and audio parts. Notice that we are not finetuning vision layer. This will save our computation. We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 8,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,               # And LoftQ
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",

        # Audio layers
        "post", "linear_start", "linear_end",
        "embedding_projection",
    ],
    modules_to_save=[
        "lm_head",
        "embed_tokens",
        "embed_audio",
    ],
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


<a name="Data"></a>
### Data Prep
We adapt the `mozilla-foundation/common_voice_17_0` dataset for our German ASR task using Gemma 3N multi-modal chat format. Each audio-text pair is structured into a conversation with `system`, `user`, and `assistant` roles. The processor then converts this into the final training format:

```
<bos><start_of_turn>system
You are an assistant that transcribes speech accurately.<end_of_turn>
<start_of_turn>user
<audio>Please transcribe this audio.<end_of_turn>
<start_of_turn>model
Ich, ich rechne direkt mich an.<end_of_turn>
```

In [ ]:
def format_intersection_data(samples: dict) -> dict[str, list]:
    """Format intersection dataset to match expected message format"""
    formatted_samples = {"messages": []}
    for idx in range(len(samples["audio"])):
        audio = samples["audio"][idx]["array"]
        label = str(samples["sentence"][idx])

        message = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": "You are an assistant that transcribes speech accurately.",
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": audio},
                    {"type": "text", "text": "Please transcribe this audio."}
                ]
            },
            {
                "role": "assistant",
                "content":[{"type": "text", "text": label}]
            }
        ]
        formatted_samples["messages"].append(message)
    return formatted_samples

In [ ]:
dataset = dataset.map(format_intersection_data, batched=True, batch_size=4, num_proc=4)

Map (num_proc=4):   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
def collate_fn(examples):
        texts = []
        audios = []

        for example in examples:
            # Apply chat template to get text
            text = processor.apply_chat_template(
                example["messages"], tokenize=False, add_generation_prompt=False
            ).strip()
            texts.append(text)

            # Extract audios
            audios.append(example["audio"]["array"])

        # Tokenize the texts and process the images
        batch = processor(
            text=texts, audio=audios, return_tensors="pt", padding=True
        )

        # The labels are the input_ids, and we mask the padding tokens in the loss computation
        labels = batch["input_ids"].clone()

        # Use Gemma3n specific token masking
        labels[labels == processor.tokenizer.pad_token_id] = -100
        if hasattr(processor.tokenizer, 'image_token_id'):
            labels[labels == processor.tokenizer.image_token_id] = -100
        if hasattr(processor.tokenizer, 'audio_token_id'):
            labels[labels == processor.tokenizer.audio_token_id] = -100
        if hasattr(processor.tokenizer, 'boi_token_id'):
            labels[labels == processor.tokenizer.boi_token_id] = -100
        if hasattr(processor.tokenizer, 'eoi_token_id'):
            labels[labels == processor.tokenizer.eoi_token_id] = -100


        batch["labels"] = labels
        return batch

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We train for one full epoch (num_train_epochs=1) to get a meaningful result.

In [ ]:
from trl import SFTTrainer, SFTConfig


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=processor.tokenizer,
    data_collator=collate_fn,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 1,
        # use reentrant checkpointing
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        warmup_ratio = 0.1,
        #max_steps = 60,
        num_train_epochs = 1,          # Set this instead of max_steps for full training runs
        learning_rate = 5e-5,
        logging_steps = 10,
        save_strategy="steps",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",             # For Weights and Biases

        # You MUST put the below items for audio finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 2,
        max_seq_length = 2048,
    )
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
10.988 GB of memory reserved.


# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
# Making sure the data format is right
display(dataset.to_pandas().head())

,client_id,path,audio,sentence,up_votes,down_votes,age,gender,accent,locale,segment,variant,messages
0,6ac0427a8caef565e3a800e844451a5dde7df70a144fae...,/root/.cache/huggingface/datasets/downloads/ex...,{'bytes': b'RIFF$.\x02\x00WAVEfmt \x10\x00\x00...,Saya mendengarkan cerita membosankan dari tema...,2,0,,,,id,,,"[{'content': [{'audio': None, 'text': 'You are..."
1,6ac0427a8caef565e3a800e844451a5dde7df70a144fae...,/root/.cache/huggingface/datasets/downloads/ex...,{'bytes': b'RIFF$\x17\x01\x00WAVEfmt \x10\x00\...,halo dunia!,2,0,,,,id,,,"[{'content': [{'audio': None, 'text': 'You are..."
2,6ac0427a8caef565e3a800e844451a5dde7df70a144fae...,/root/.cache/huggingface/datasets/downloads/ex...,{'bytes': b'RIFF$\xa7\x01\x00WAVEfmt \x10\x00\...,Sudah makan? sudah sholat...?,2,0,,,,id,,,"[{'content': [{'audio': None, 'text': 'You are..."
3,6ac0427a8caef565e3a800e844451a5dde7df70a144fae...,/root/.cache/huggingface/datasets/downloads/ex...,{'bytes': b'RIFF$M\x01\x00WAVEfmt \x10\x00\x00...,mau pergi kemana hari ini?,2,0,,,,id,,,"[{'content': [{'audio': None, 'text': 'You are..."
4,6ac0427a8caef565e3a800e844451a5dde7df70a144fae...,/root/.cache/huggingface/datasets/downloads/ex...,{'bytes': b'RIFF$;\x01\x00WAVEfmt \x10\x00\x00...,udah keluar hasil testnya?,2,0,,,,id,,,"[{'content': [{'audio': None, 'text': 'You are..."


In [ ]:
sample = dataset[0]
processed_sample = collate_fn([sample])
print("Sample Text:", sample['messages'][2]['content'][0]['text'])
print("Processed Input IDs:", processed_sample['input_ids'])

Sample Text: Saya mendengarkan cerita membosankan dari teman saya.
Processed Input IDs: tensor([[     2,      2,    105,   2364,    107,   3048,    659,    614,  16326,
            600,   1207,  73309,  10808,  23704, 236761,    110, 256000, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273, 262273,
         262273, 2622

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 12,546,048 of 5,451,984,320 (0.23% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,10.696100
20,10.782300
30,2.062500
40,0.975000
50,0.705000
60,0.587900
70,0.612400
80,0.467800
90,0.507300
100,0.469100


Unsloth: Will smartly offload gradients to save VRAM!


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

523.1184 seconds used for training.
8.72 minutes used for training.
Peak reserved memory = 10.988 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 74.54 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64` but for this example we use `do_sample=False` for ASR.

In [ ]:
messages = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": "You are an assistant that transcribes speech accurately.",
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": test_audio['audio']['array']},
                    {"type": "text", "text": "Please transcribe this audio."}
                ]
            }
        ]

do_gemma_3n_inference(messages, max_new_tokens = 256)

Oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh,


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma-3n")  # Local saving
processor.save_pretrained("gemma-3n")
model.push_to_hub("KronosDP/gemma-3n-id-4bit-300v", token = "") # Online saving
# processor.push_to_hub("HF_ACCOUNT/gemma-3n", token = "...") # Online saving

README.md:   0%|          | 0.00/605 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/50.3M [00:00<?, ?B/s]

Saved model to https://huggingface.co/KronosDP/gemma-3n-id-4bit-300v


Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, processor = FastModel.from_pretrained(
        model_name = "gemma-3n", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-3N?",}]
}]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(processor, skip_prompt = True),
)